In [9]:
# 开启 AutoDL 的学术加速（专门加速 Github 和 Kaggle）
!source /etc/network_turbo && kaggle competitions download -c state-farm-distracted-driver-detection

# 如果要关闭加速，可以使用：
# !source /etc/network_turbo_disable

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证
100%|██████████████████████████████████████| 4.00G/4.00G [07:52<00:00, 9.10MB/s]



In [6]:
# 1. 安装核心依赖（修正了 grad-cam 的包名）
!pip install -q kaggle ultralytics timm albumentations grad-cam pandas seaborn transformers

# 2. 配置 Kaggle 秘钥
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. 创建数据集目录并下载
!mkdir -p ./dataset
%cd ./dataset
!kaggle competitions download -c state-farm-distracted-driver-detection

# 4. 静默解压并清理压缩包
!unzip -q state-farm-distracted-driver-detection.zip
!rm state-farm-distracted-driver-detection.zip
%cd ..

print("✅ 环境配置与数据下载完毕！")

/root/dataset
100%|██████████████████████████████████████| 4.00G/4.00G [12:13<00:00, 5.85MB/s]

/root
✅ 环境配置与数据下载完毕！


In [7]:
import torch

print("=========================================")
print("          🖥️ 深度学习环境体检报告          ")
print("=========================================")
print(f"📦 PyTorch 版本: {torch.__version__}")
print(f"🚀 CUDA 是否可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"😎 识别到的显卡: {torch.cuda.get_device_name(0)}")
    print(f"⚙️ 当前 CUDA 版本: {torch.version.cuda}")
    print(f"🧠 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 警告：PyTorch 无法调用 GPU，请检查服务器的 CUDA 驱动！")
print("=========================================")

          🖥️ 深度学习环境体检报告          
📦 PyTorch 版本: 2.8.0+cu128
🚀 CUDA 是否可用: True
😎 识别到的显卡: NVIDIA GeForce RTX 5090
⚙️ 当前 CUDA 版本: 12.8
🧠 显存大小: 31.36 GB


In [1]:
import os
import cv2
from tqdm.notebook import tqdm
from ultralytics import YOLO
import numpy as np

SOURCE_DIR = './dataset/imgs/train'
TARGET_DIR = './dataset/imgs/train_cropped_v2'

print("🚀 正在加载 YOLOv8n 检测模型...")
model = YOLO('yolov8n.pt')

for c in range(10):
    os.makedirs(os.path.join(TARGET_DIR, f'c{c}'), exist_ok=True)

all_images = []
for c in range(10):
    class_dir = os.path.join(SOURCE_DIR, f'c{c}')
    if not os.path.exists(class_dir): continue
    for img_name in os.listdir(class_dir):
        all_images.append((f'c{c}', img_name))

print(f"📊 共找到 {len(all_images)} 张图片，5090 准备就绪，开始极速裁剪...")

no_detect_count = 0
for class_name, img_name in tqdm(all_images, desc="裁剪进度"):
    img_path = os.path.join(SOURCE_DIR, class_name, img_name)
    save_path = os.path.join(TARGET_DIR, class_name, img_name)

    if os.path.exists(save_path): continue

    img = cv2.imread(img_path)
    if img is None: continue
    h, w = img.shape[:2]

    # 让 GPU 0 (你的 5090) 参与推理
    results = model.predict(img, classes=[0], conf=0.3, verbose=False, device=0)

    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        best_idx = np.argmax(areas)

        x1, y1, x2, y2 = boxes[best_idx].astype(int)
        person_h = y2 - y1
        new_y2 = y1 + int(person_h * 0.65)
        pad_x = int((x2 - x1) * 0.15) 
        pad_y_top = int(person_h * 0.05)   

        crop_x1 = max(0, x1) 
        crop_y1 = max(0, y1 - pad_y_top)
        crop_x2 = min(w, x2 + pad_x)
        crop_y2 = min(h, new_y2) 

        if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
             crop_img = img[y1:y2, x1:x2]
        else:
             crop_img = img[crop_y1:crop_y2, crop_x1:crop_x2]

        final_img = cv2.resize(crop_img, (384, 384), interpolation=cv2.INTER_LINEAR)
        cv2.imwrite(save_path, final_img)
    else:
        no_detect_count += 1
        final_img = cv2.resize(img, (384, 384), interpolation=cv2.INTER_LINEAR)
        cv2.imwrite(save_path, final_img)

print(f"\n🎉 智能裁剪完成！")

🚀 正在加载 YOLOv8n 检测模型...
📊 共找到 0 张图片，5090 准备就绪，开始极速裁剪...


裁剪进度: 0it [00:00, ?it/s]


🎉 智能裁剪完成！


In [4]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [2]:
#快速调试代码
import os
import ssl
import requests

# 禁用 SSL 证书验证（用于学术加速的自签名证书）
ssl._create_default_https_context = ssl._create_unverified_context
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
requests.packages.urllib3.disable_warnings()

import os
import cv2
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import log_loss, confusion_matrix
import timm
from transformers import get_cosine_schedule_with_warmup

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


# ==========================================
# ⚙️ 1. 全局配置参数 (A100 优化版)
# ==========================================
CSV_PATH = 'dataset/driver_imgs_list.csv'
TRAIN_DIR = 'dataset/imgs/train_cropped_v2' # 去背景后的图片集

MODEL_NAME = 'swin_base_patch4_window12_384.ms_in22k'
IMG_SIZE = 384
EPOCHS = 10         # ✅ 正式训练，设为 10 轮 (配合早停机制)
BATCH_SIZE = 32    # ✅ A100 显存极大，物理 Batch Size 提升到 32
ACCUMULATION_STEPS = 1  # ✅ 不需要频繁梯度累加，直接真实更新
NUM_SPLITS = 5
TARGET_FOLDS = [0,1,2,3,4]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 清理内存环境
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 📊 2. 数据划分 (完全复刻 Eval_Pipeline 贪心隔离策略)
# ==========================================
def generate_balanced_folds(csv_path, n_splits=5):
    df = pd.read_csv(csv_path)
    df = df.reset_index(drop=True)

    if 'label_int' not in df.columns:
        df['label_int'] = df['classname'].str.extract(r'(\d+)').astype(int)

    # 贪心算法逻辑：按每个驾驶员的图片数从大到小排序
    driver_counts = df.groupby('subject').size().sort_values(ascending=False)

    fold_totals = np.zeros(n_splits)
    fold_groups = [[] for _ in range(n_splits)]

    # 每次将最大的驾驶员分配给当前图片数最少的 Fold
    for subject, count in driver_counts.items():
        min_fold_idx = np.argmin(fold_totals)
        fold_groups[min_fold_idx].append(subject)
        fold_totals[min_fold_idx] += count

    # 映射回原 DataFrame
    df['fold'] = -1
    for i, subjects in enumerate(fold_groups):
        df.loc[df['subject'].isin(subjects), 'fold'] = i

    # 打印隔离与平衡性检查报告
    print("\n" + "="*50)
    print("           📊 5 折数据分布情况")
    print("="*50)
    for i in range(n_splits):
        num_driver = len(fold_groups[i])
        num_img = int(fold_totals[i])
        print(f"Fold {i}  |  驾驶员数量: {num_driver:2d}  |  图片总数: {num_img:4d}")
    print("="*50)
    print(f"✅ 最大样本偏差: {int(fold_totals.max() - fold_totals.min())} 张图\n")

    output_path = "train_with_folds.csv"
    df.to_csv(output_path, index=False)
    print("\n✅ 划分完成: train_with_folds.csv\n")

    return df

# ==========================================
# 🖼️ 3. 数据集与增强
# ==========================================
def get_train_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0, p=0.6),
        A.Affine(translate_percent=(-0.05, 0.05), scale=(0.95, 1.05), rotate=(-10, 10), p=0.5),
        A.GaussNoise(p=0.4),
        #A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 16), hole_width_range=(1, 16), fill=0, p=0.4),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

class DriverDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['classname'], row['img'])

        image = cv2.imread(img_path)
        if image is None:
            # 防御性回退：如果没找到抠图后的文件，去原图里拿
            fallback_path = os.path.join('/content/dataset/imgs/train', row['classname'], row['img'])
            image = cv2.imread(fallback_path)
            if image is None:
                raise FileNotFoundError(f"图片丢失: {img_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)['image']

        return image, row['label_int']

# ==========================================
# 🛠️ 4. 评估工具 (GradCAM & 混淆矩阵)
# ==========================================
def plot_confusion_matrix(y_true, y_pred, fold_idx, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues')
    plt.title(f'Fold {fold_idx} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(save_dir, f'cm_fold_{fold_idx}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def generate_grad_cam(model, val_loader, fold_idx, save_dir):
    model.eval()
    try:
        images, labels = next(iter(val_loader))
    except StopIteration:
        return

    target_layer = model.layers[-1].blocks[-1].norm1

    def reshape_transform(tensor, height=12, width=12):
        if tensor.ndim == 4:
            return tensor.permute(0, 3, 1, 2)
        elif tensor.ndim == 3:
            result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
            return result.permute(0, 3, 1, 2)

    cam = GradCAM(model=model, target_layers=[target_layer], reshape_transform=reshape_transform)
    input_tensor = images[0:1].to(DEVICE)

    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0, :]
    img_np = images[0].permute(1, 2, 0).cpu().numpy()

    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    img_np = np.clip(std * img_np + mean, 0, 1)

    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    plt.imsave(os.path.join(save_dir, f"grad_cam_fold_{fold_idx}.png"), cam_image)

# ==========================================
# 🚀 5. 主干训练流程
# ==========================================
def main():
    base_dir = 'models'
    os.makedirs(base_dir, exist_ok=True)

    full_df = generate_balanced_folds(CSV_PATH, NUM_SPLITS)
    oof_preds = np.zeros((len(full_df), 10))

    for fold in TARGET_FOLDS:
        print(f"\n{'='*40}\n🌟 开始训练 Fold {fold} 🌟\n{'='*40}")

        train_df = full_df[full_df['fold'] != fold]
        val_df = full_df[full_df['fold'] == fold]

        train_loader = DataLoader(
            DriverDataset(train_df, TRAIN_DIR, transform=get_train_transforms(IMG_SIZE)),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True # ✅ num_workers=4 提速
        )
        val_loader = DataLoader(
            DriverDataset(val_df, TRAIN_DIR, transform=get_valid_transforms(IMG_SIZE)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True # ✅ num_workers=4 提速
        )
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10, drop_path_rate=0.3)
    
    # 2. 从本地手动读取刚刚下载的权重
        from safetensors.torch import load_file
        state_dict = load_file("swin_base.safetensors")
        
        # 3. 剥离原权重中的分类头（原模型是 2万类，我们只要 10 类，防止形状冲突报错）
        for key in list(state_dict.keys()):
            if key.startswith('head.'):
                del state_dict[key]
                
        # 4. 把纯净的权重塞进模型里
        model.load_state_dict(state_dict, strict=False)
        model.to(DEVICE)

        class_weights=torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0],dtype=torch.float).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights,label_smoothing=0.1)
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
        scaler = torch.amp.GradScaler('cuda')

        total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_steps * 0.1),
            num_training_steps=total_steps
        )

        best_val_loss = float('inf')
        save_path = os.path.join(base_dir, f"best_model_swin_fold_{fold}.pth")

        EARLY_STOP_PATIENCE = 2
        epochs_no_improve = 0

        for epoch in range(EPOCHS):
            model.train()
            optimizer.zero_grad()
            train_loss = 0.0

            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
            for i, (images, labels) in enumerate(pbar):
                images, labels = images.to(DEVICE), labels.to(DEVICE)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels) / ACCUMULATION_STEPS

                scaler.scale(loss).backward()
                train_loss += loss.item() * ACCUMULATION_STEPS

                if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                pbar.set_postfix({'loss': f"{loss.item()*ACCUMULATION_STEPS:.4f}"})

            model.eval()
            val_loss = 0.0
            fold_preds_list = []
            fold_labels = []

            vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
            with torch.no_grad():
                for images, labels in vbar:
                    images, labels = images.to(DEVICE), labels.to(DEVICE)

                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    fold_preds_list.append(outputs.softmax(dim=1).cpu().numpy())
                    fold_labels.extend(labels.cpu().numpy())
                    vbar.set_postfix({'v_loss': f"{loss.item():.4f}"})

            avg_val_loss = val_loss / len(val_loader)
            current_fold_preds = np.concatenate(fold_preds_list, axis=0)

            if avg_val_loss < best_val_loss:
                print(f"✅ 验证集 Loss 降低: {best_val_loss:.4f} -> {avg_val_loss:.4f}，保存模型。")
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), save_path)
                epochs_no_improve = 0  # 重置计数器

                oof_preds[val_df.index] = current_fold_preds
                best_labels = fold_labels
                best_preds = np.argmax(current_fold_preds, axis=1)
            else:
                epochs_no_improve += 1
                print(f"⚠️ 验证集 Loss 未降低 ({epochs_no_improve}/{EARLY_STOP_PATIENCE})")
                if epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print(f"🛑 连续 {EARLY_STOP_PATIENCE} 轮 Loss 未下降，触发早停机制 (Early Stopping)，提前结束本折训练！")
                    break

        print("📸 正在生成混淆矩阵与 Grad-CAM 注意力热力图...")
        plot_confusion_matrix(best_labels, best_preds, fold, base_dir)

        model.load_state_dict(torch.load(save_path))
        generate_grad_cam(model, val_loader, fold, base_dir)

        del model, optimizer, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    print("\n🎉 所有 Fold 训练完毕！保存 OOF 预测结果...")
    np.save(os.path.join(base_dir, "oof_preds_swin.npy"), oof_preds)

    final_labels = full_df['label_int'].values
    final_log_loss = log_loss(final_labels, oof_preds)
    print(f"🔥 Final OOF Cross-Validation Log Loss: {final_log_loss:.4f}")

if __name__ == "__main__":
    main()

    # ✅ 训练完成后自动断开实例以节省算力
    print("\n🛑 训练全部结束，模型已保存至 Google Drive。正在自动断开实例...")
   



           📊 5 折数据分布情况
Fold 0  |  驾驶员数量:  5  |  图片总数: 4407
Fold 1  |  驾驶员数量:  5  |  图片总数: 4475
Fold 2  |  驾驶员数量:  5  |  图片总数: 4364
Fold 3  |  驾驶员数量:  6  |  图片总数: 4704
Fold 4  |  驾驶员数量:  5  |  图片总数: 4474
✅ 最大样本偏差: 340 张图


✅ 划分完成: train_with_folds.csv


🌟 开始训练 Fold 0 🌟


Epoch 1/10 [Train]:   0%|          | 0/563 [00:00<?, ?it/s]/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Epoch 1/10 [Valid]: 100%|██████████| 138/138 [00:09<00:00, 14.55it/s, v_loss=2.7523]


✅ 验证集 Loss 降低: inf -> 0.8087，保存模型。


Epoch 2/10 [Valid]: 100%|██████████| 138/138 [00:09<00:00, 14.56it/s, v_loss=1.1310]


✅ 验证集 Loss 降低: 0.8087 -> 0.7431，保存模型。


...

Epoch 3/10 [Valid]: 100%|██████████| 140/140 [00:09<00:00, 14.35it/s, v_loss=0.9025]


⚠️ 验证集 Loss 未降低 (1/2)


Epoch 4/10 [Valid]: 100%|██████████| 140/140 [00:09<00:00, 14.48it/s, v_loss=0.8869]


⚠️ 验证集 Loss 未降低 (2/2)
🛑 连续 2 轮 Loss 未下降，触发早停机制 (Early Stopping)，提前结束本折训练！
📸 正在生成混淆矩阵与 Grad-CAM 注意力热力图...

🎉 所有 Fold 训练完毕！保存 OOF 预测结果...
🔥 Final OOF Cross-Validation Log Loss: 0.3200

🛑 训练全部结束，模型已保存至 Google Drive。正在自动断开实例...


/root/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [7]:
import os
import timm
from huggingface_hub import constants
constants.ENDPOINT = "https://hf-mirror.com"
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 验证环境变量
print("HF_ENDPOINT =", os.environ.get('HF_ENDPOINT'))

# 尝试用 huggingface_hub 库直接请求镜像站
from huggingface_hub import hf_hub_url
url = hf_hub_url("timm/swin_base_patch4_window12_384.ms_in22k", "model.safetensors")
print("实际请求 URL:", url)

HF_ENDPOINT = https://hf-mirror.com
实际请求 URL: https://huggingface.co/timm/swin_base_patch4_window12_384.ms_in22k/resolve/main/model.safetensors


In [2]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [4]:
# --no-check-certificate 这个参数是核武器，强行无视任何证书拦截！
!wget -q --show-progress --no-check-certificate https://hf-mirror.com/timm/swin_base_patch4_window12_384.ms_in22k/resolve/main/model.safetensors -O swin_base.safetensors
print("✅ 预训练权重下载完成！")

swin_base.safetenso 100%[===================>] 429.82M  12.7MB/s    in 41s     
✅ 预训练权重下载完成！


In [3]:
import os
import torch
import pandas as pd
import numpy as np
import timm
from tqdm.notebook import tqdm  # 保持 Jupyter 的漂亮进度条
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# ⚙️ 1. 基础配置 (RTX 5090 本地路径版)
# ==========================================
# 🔥 5090 专属加速魔法：开启 TF32 核心
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
folds=[0,1,2,3,4]
# 严格按照你本地的相对路径设置
SAMPLE_SUBMISSION_PATH = './dataset/sample_submission.csv'
TEST_DIR = 'dataset/imgs/test_cropped_v2' 
SAVE_DIR = './models'

# 模型与权重配置
MODEL_NAME = 'swin_base_patch4_window12_384.ms_in22k'
WEIGHT_PATH_TEMPLATE = os.path.join(SAVE_DIR, 'best_model_swin_fold_{}.pth')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 384
BATCH_SIZE = 128  # 5090 显存有 32G，推理不存梯度，直接开到 128 起飞
NUM_WORKERS = 8

# ==========================================
# 🖼️ 2. 构建测试集 DataLoader (按 CSV 顺序严格读取)
# ==========================================
class TestDriverDataset(Dataset):
    def __init__(self, csv_path, test_dir):
        # 🌟 关键修复：直接读取官方样本提交文件，严格保证顺序一致性
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"🚨 找不到官方提交模板: {csv_path}")
        self.df = pd.read_csv(csv_path)
        self.test_dir = test_dir
        
        # 提取所有的图片文件名 (形如 img_1.jpg)
        self.image_names = self.df['img'].values

        # 测试集的预处理必须和验证集一模一样！仅做标准化，无随机增强
        self.transform = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.test_dir, img_name)
        
        image = cv2.imread(img_path)
        if image is None:
             raise FileNotFoundError(f"找不到图片: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # ⚠️ 删除了所有的裁剪逻辑，直接把原图喂进去缩放
        augmented = self.transform(image=image)
        image_tensor = augmented['image']

        return image_tensor, img_name

# ==========================================
# 🚀 3. 核心预测与 5折融合 (Ensemble) 逻辑
# ==========================================
def generate_submission():
    print(f"🚀 启动预测流程！使用设备: {DEVICE}")

    # 1. 准备测试数据
    test_dataset = TestDriverDataset(SAMPLE_SUBMISSION_PATH, TEST_DIR)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print(f"📦 严格按照 CSV 顺序，共抓取 {len(test_dataset)} 张测试图片。")

    # 2. 准备空容器存放预测结果
    all_fold_preds = []

    # 3. 实例化空模型壳子
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10)
    model.to(DEVICE)
    model.eval()

    # 如果你训练时用了 torch.compile，这里最好也开启，以保证权重结构的完美兼容
    if int(torch.__version__.split('.')[0]) >= 2: 
        print("⚡ 启用 torch.compile 加速推理...")
        model = torch.compile(model)

    # 4. 依次请出 5 折模型进行预测
    for fold in folds:
        # 1. 每次循环重新创建一个干净的空模型
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10)
        model.to(DEVICE)
        model.eval()

        weight_path = WEIGHT_PATH_TEMPLATE.format(fold)
        print(f"\n👨‍⚖️ 正在加载第 {fold} 号权重 ({weight_path})...")

        if not os.path.exists(weight_path):
            continue

        # 2. 读取权重（由于训练时已经去掉了前缀，这里直接读出来的就是干净的）
        state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
        
        # 3. 🎯 关键修复：先加载权重，并且去掉 strict=False 以防万一
        model.load_state_dict(state_dict, strict=True) 
        
        # 4. 🎯 关键修复：权重安全装填完毕后，再进行图编译加速
        if int(torch.__version__.split('.')[0]) >= 2:
            print("⚡ 启用 torch.compile 加速推理...")
            model = torch.compile(model)

        fold_preds = []
        with torch.no_grad():
             with torch.amp.autocast('cuda'):
                pbar = tqdm(test_loader, desc=f"Fold {fold} 预测中")
                for images, _ in pbar:
                    images = images.to(DEVICE)
                    outputs = model(images)
                    probs = torch.softmax(outputs, dim=1)
                    fold_preds.append(probs.cpu().numpy())

        fold_preds = np.concatenate(fold_preds, axis=0)
        all_fold_preds.append(fold_preds)

    if not all_fold_preds:
        print("❌ 没有找到任何有效的权重文件，推理终止。")
        return

    # 5. 终极平均融合
    print(f"\n🤝 正在对找到的 {len(all_fold_preds)} 个模型进行概率平均融合...")
    all_fold_preds = np.array(all_fold_preds) 
    final_preds = np.mean(all_fold_preds, axis=0) 

    # 6. 生成 Kaggle 要求的 submission.csv
    print("📝 正在生成最终的 CSV 提交文件...")
    
    # 直接复用 DataLoader 里的准确顺序
    img_filenames = test_dataset.image_names

    df_submit = pd.DataFrame(final_preds, columns=[f'c{i}' for i in range(10)])
    df_submit.insert(0, 'img', img_filenames) 

    submit_filename = os.path.join(SAVE_DIR, 'submission_swin_5fold_fixed.csv')
    df_submit.to_csv(submit_filename, index=False)
    print(f"🎉 大功告成！预测文件已安全保存为: {submit_filename}")

if __name__ == '__main__':
    generate_submission()

🚀 启动预测流程！使用设备: cuda
📦 严格按照 CSV 顺序，共抓取 79726 张测试图片。
⚡ 启用 torch.compile 加速推理...

👨‍⚖️ 正在加载第 0 号权重 (./models/best_model_swin_fold_0.pth)...
⚡ 启用 torch.compile 加速推理...


Fold 0 预测中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载第 1 号权重 (./models/best_model_swin_fold_1.pth)...
⚡ 启用 torch.compile 加速推理...


Fold 1 预测中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载第 2 号权重 (./models/best_model_swin_fold_2.pth)...
⚡ 启用 torch.compile 加速推理...


...


👨‍⚖️ 正在加载第 3 号权重 (./models/best_model_swin_fold_3.pth)...
⚡ 启用 torch.compile 加速推理...


Fold 3 预测中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载第 4 号权重 (./models/best_model_swin_fold_4.pth)...
⚡ 启用 torch.compile 加速推理...


Fold 4 预测中:   0%|          | 0/623 [00:00<?, ?it/s]


🤝 正在对找到的 5 个模型进行概率平均融合...
📝 正在生成最终的 CSV 提交文件...
🎉 大功告成！预测文件已安全保存为: ./models/submission_swin_5fold_fixed.csv
